# Notebook 7: Econometric Results and Diagnostics

**Research Project:** Improving Asymmetric Exchange Rate Pass-Through Modelling Across Food Price Categories in South Africa Using Machine Learning

**Research Period:** April 2017 – December 2025

## Notebook Objective

This notebook evaluates the statistical adequacy and reliability of the econometric models developed in Notebook 6.

The analysis focuses on:

1. reproducing the selected ARDL and NARDL specifications
2. assessing residual serial correlation
3. testing for heteroskedasticity
4. examining residual normality
5. evaluating parameter and model stability
6. reviewing models with diagnostic failures
7. consolidating the long-run and short-run findings
8. exporting validated econometric results for later comparison with the machine learning models.

No new lag selection is performed in this notebook. The specifications selected
in Notebook 6 are treated as the starting point for diagnostic assessment.

## Diagnostic Context

Statistically significant coefficients do not necessarily imply that a model is adequately specified.

ARDL and NARDL inference depends on assumptions concerning the behaviour of the model residuals and the stability of the estimated relationship. Diagnostic testing is therefore required before the estimated pass-through effects are treated as final research findings.

The diagnostic process distinguishes between:

- a statistically estimated relationship;
- a relationship that passes the relevant diagnostic tests; and
- a relationship that remains useful but requires a methodological caveat.

This distinction prevents coefficient significance from being interpreted without considering the reliability of the underlying model.

## Diagnostic Framework

The selected econometric models are assessed using the following diagnostic areas:

### Serial correlation

Residual serial correlation indicates that the model has not fully captured the time-dependent structure of the series.

### Heteroskedasticity

Heteroskedastic residuals have non-constant variance and may affect the
reliability of conventional standard errors and hypothesis tests.

### Residual normality

Normality is assessed because strongly non-normal residuals may influence small-sample statistical inference. Normality is treated as a supporting diagnostic rather than an automatic model-rejection rule.

### Functional form

Functional-form assessment examines whether important nonlinear structure may remain unexplained by the model.

### Parameter stability

Stability tests assess whether the estimated relationship remains reasonably consistent across the modelling period.

A 5% significance level is used unless otherwise stated. Diagnostic failures are reported transparently and considered jointly rather than using one test as an automatic reason to discard a model.

In [28]:
# import required libraries
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from statsmodels.tsa.ardl import ARDL, UECM
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


## Load Econometric Data and Results

The category-level econometric dataset and the result tables exported by Notebook 6 are loaded from their established project locations.

The exported lag-selection tables provide the specifications required to re-estimate the models. The remaining tables preserve the bounds-test, error-correction, coefficient and asymmetry findings that will be evaluated against the diagnostic results.

In [29]:
# define input locations
econometric_data_path = Path(
    "../data/processed/econometric_model_data.csv"
)

econometric_results_directory = Path(
    "../reports/tables/econometrics"
)

result_file_names = [
    "symmetric_lag_selection.csv",
    "asymmetric_lag_selection.csv",
    "long_run_bic_comparison.csv",
    "final_bounds_results.csv",
    "error_correction_results.csv",
    "long_run_effects.csv",
    "long_run_asymmetry_results.csv",
    "symmetric_short_run_selection.csv",
    "asymmetric_short_run_selection.csv",
    "short_run_bic_comparison.csv",
    "short_run_asymmetry_results.csv",
]

required_paths = [
    econometric_data_path,
    *[
        econometric_results_directory / file_name
        for file_name in result_file_names
    ],
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    missing_path_list = "\n".join(
        str(path) for path in missing_paths
    )
    raise FileNotFoundError(
        f"Required Notebook 6 outputs are missing:\n{missing_path_list}"
    )

print("All required input files are available.")

All required input files are available.


In [30]:
# load the econometric dataset
econometric_data = pd.read_csv(
    econometric_data_path,
    parse_dates=["Date"],
)

# Load the exported result tables
econometric_result_tables = {
    Path(file_name).stem: pd.read_csv(
        econometric_results_directory / file_name
    )
    for file_name in result_file_names
}

symmetric_lag_selection = econometric_result_tables[
    "symmetric_lag_selection"
]
asymmetric_lag_selection = econometric_result_tables[
    "asymmetric_lag_selection"
]
final_bounds_results = econometric_result_tables[
    "final_bounds_results"
]
error_correction_results = econometric_result_tables[
    "error_correction_results"
]
long_run_effects = econometric_result_tables[
    "long_run_effects"
]
long_run_asymmetry_results = econometric_result_tables[
    "long_run_asymmetry_results"
]
symmetric_short_run_selection = econometric_result_tables[
    "symmetric_short_run_selection"
]
asymmetric_short_run_selection = econometric_result_tables[
    "asymmetric_short_run_selection"
]
short_run_asymmetry_results = econometric_result_tables[
    "short_run_asymmetry_results"
]

print("Econometric data and result tables loaded successfully.")

Econometric data and result tables loaded successfully.


In [31]:
# validate the Notebook 6 handoff
input_validation = pd.Series(
    {
        "Econometric observations": len(econometric_data),
        "Food subclasses": econometric_data[
            "SubclassDescription"
        ].nunique(),
        "Unique months": econometric_data["Date"].nunique(),
        "Duplicate subclass-month rows": econometric_data.duplicated(
            subset=["SubclassDescription", "Date"]
        ).sum(),
        "Missing econometric values": int(
            econometric_data.isna().sum().sum()
        ),
        "Result tables loaded": len(econometric_result_tables),
        "Symmetric specifications": len(
            symmetric_lag_selection
        ),
        "Asymmetric specifications": len(
            asymmetric_lag_selection
        ),
        "Final bounds-test results": len(
            final_bounds_results
        ),
    },
    name="Value",
).to_frame()

display(input_validation)

result_table_manifest = pd.DataFrame(
    [
        {
            "Table": table_name,
            "Rows": result_table.shape[0],
            "Columns": result_table.shape[1],
        }
        for table_name, result_table
        in econometric_result_tables.items()
    ]
)

display(result_table_manifest)

,Value
Econometric observations,4830
Food subclasses,46
Unique months,105
Duplicate subclass-month rows,0
Missing econometric values,0
Result tables loaded,11
Symmetric specifications,46
Asymmetric specifications,46
Final bounds-test results,92


,Table,Rows,Columns
0,symmetric_lag_selection,46,7
1,asymmetric_lag_selection,46,7
2,long_run_bic_comparison,46,11
3,final_bounds_results,92,12
4,error_correction_results,9,9
5,long_run_effects,14,9
6,long_run_asymmetry_results,6,13
7,symmetric_short_run_selection,46,7
8,asymmetric_short_run_selection,46,7
9,short_run_bic_comparison,46,11


## Re-estimate Selected Models

The selected models are re-estimated using the lag orders exported by Notebook 6.

Four model collections are reconstructed:

1. symmetric ARDL level models
2. asymmetric NARDL level models
3. symmetric stationary short-run models
4. asymmetric stationary short-run models

The same dependent variables, exchange-rate variables, seasonal indicators and six-month hold-back period used during model selection are retained. This ensures that the diagnostic tests are applied to the exact specifications from which the reported results were obtained.

Re-estimation also makes the model residuals and fitted values available within the current notebook without relying on temporary Python objects from Notebook 6.

### Re-estimate the Level Models

The selected symmetric ARDL and asymmetric NARDL level models are reconstructed from their exported lag specifications.

For every food subclass, the reconstruction retains:

- log CPI as the dependent variable
- the selected food-price lag order
- the selected exchange-rate or component lag order
- a constant
- monthly seasonal indicators
- a seasonal period of 12 months
- the common six-month hold-back period

The re-estimated BIC values are compared with the values exported by Notebook 6. Matching BIC values confirm that the same model specifications have been reproduced.

In [32]:
def fit_selected_level_models(
    data,
    selection_table,
    exogenous_columns,
    exchange_rate_lag_column,
):
    """Re-estimate selected level ARDL and UECM models."""

    ardl_results = {}
    uecm_results = {}
    validation_records = []

    for selection_row in selection_table.itertuples(index=False):
        subclass = selection_row.SubclassDescription
        price_lag = int(selection_row.Price_Lag)
        exchange_rate_lag = int(
            getattr(selection_row, exchange_rate_lag_column)
        )

        subclass_data = (
            data.loc[
                data["SubclassDescription"].eq(subclass)
            ]
            .sort_values("Date")
            .set_index("Date")
            .asfreq("MS")
        )

        ardl_model = ARDL(
            endog=subclass_data["Log_CPI"],
            lags=price_lag,
            exog=subclass_data[exogenous_columns],
            order=exchange_rate_lag,
            trend="c",
            seasonal=True,
            period=12,
            causal=False,
            hold_back=6,
            missing="raise",
        )

        ardl_result = ardl_model.fit()
        uecm_result = UECM.from_ardl(
            ardl_result.model
        ).fit()

        ardl_results[subclass] = ardl_result
        uecm_results[subclass] = uecm_result

        validation_records.append(
            {
                "SubclassDescription": subclass,
                "Price_Lag": price_lag,
                "Exchange_Rate_Lag": exchange_rate_lag,
                "Exported_BIC": selection_row.BIC,
                "Reestimated_BIC": ardl_result.bic,
                "Absolute_BIC_Difference": abs(
                    selection_row.BIC - ardl_result.bic
                ),
                "BIC_Match": np.isclose(
                    selection_row.BIC,
                    ardl_result.bic,
                    rtol=1e-10,
                    atol=1e-8,
                ),
                "Observations": int(ardl_result.nobs),
            }
        )

    validation_table = (
        pd.DataFrame(validation_records)
        .sort_values("SubclassDescription")
        .reset_index(drop=True)
    )

    return ardl_results, uecm_results, validation_table

In [33]:
# re-estimate symmetric level models
(
    symmetric_level_ardl_results,
    symmetric_level_uecm_results,
    symmetric_level_validation,
) = fit_selected_level_models(
    data=econometric_data,
    selection_table=symmetric_lag_selection,
    exogenous_columns=["Log_ExchangeRate"],
    exchange_rate_lag_column="Exchange_Rate_Lag",
)

# Re-estimate asymmetric level models
(
    asymmetric_level_ardl_results,
    asymmetric_level_uecm_results,
    asymmetric_level_validation,
) = fit_selected_level_models(
    data=econometric_data,
    selection_table=asymmetric_lag_selection,
    exogenous_columns=[
        "ExchangeRate_Positive_Cumulative_Pct",
        "ExchangeRate_Negative_Cumulative_Pct",
    ],
    exchange_rate_lag_column="Component_Lag",
)

print(
    "Symmetric level models re-estimated:",
    len(symmetric_level_ardl_results),
)
print(
    "Asymmetric level models re-estimated:",
    len(asymmetric_level_ardl_results),
)

Symmetric level models re-estimated: 46
Asymmetric level models re-estimated: 46


In [34]:
# validate the reconstructed level models
symmetric_level_validation["Model"] = "ARDL"
asymmetric_level_validation["Model"] = "NARDL"

level_model_validation = pd.concat(
    [
        symmetric_level_validation,
        asymmetric_level_validation,
    ],
    ignore_index=True,
)

level_reproduction_summary = (
    level_model_validation
    .groupby("Model", observed=True)
    .agg(
        Models=("SubclassDescription", "size"),
        BIC_Matches=("BIC_Match", "sum"),
        Maximum_BIC_Difference=(
            "Absolute_BIC_Difference",
            "max",
        ),
        Minimum_Observations=("Observations", "min"),
        Maximum_Observations=("Observations", "max"),
    )
    .reset_index()
)

display(level_reproduction_summary)

print(
    "All level-model BIC values reproduced:",
    level_model_validation["BIC_Match"].all(),
)

,Model,Models,BIC_Matches,Maximum_BIC_Difference,Minimum_Observations,Maximum_Observations
0,ARDL,46,46,0.000000,99,99
1,NARDL,46,46,0.000000,99,99


All level-model BIC values reproduced: True


### 4.2 Re-estimate the Short-Run Models

The stationary symmetric and asymmetric short-run models are reconstructed from their exported lag specifications.

Monthly food-price inflation is used as the dependent variable. The symmetric models use total monthly exchange-rate changes, while the asymmetric models use separate depreciation and signed appreciation shocks.

Unlike the level models, these specifications do not require a UECM
representation because all variables enter as stationary monthly changes.

In [35]:
def fit_selected_short_run_models(
    data,
    selection_table,
    exogenous_columns,
):
    """Re-estimate selected stationary short-run models."""

    fitted_results = {}
    validation_records = []

    for selection_row in selection_table.itertuples(index=False):
        subclass = selection_row.SubclassDescription
        price_lag = int(selection_row.Price_Lag)
        exchange_rate_lag = int(
            selection_row.Exchange_Rate_Lag
        )

        subclass_data = (
            data.loc[
                data["SubclassDescription"].eq(subclass)
            ]
            .sort_values("Date")
            .set_index("Date")
        )

        exogenous_order = {
            column: exchange_rate_lag
            for column in exogenous_columns
        }

        short_run_model = ARDL(
            endog=subclass_data["Food_Inflation_Pct"],
            lags=price_lag,
            exog=subclass_data[exogenous_columns],
            order=exogenous_order,
            trend="c",
            seasonal=True,
            period=12,
            causal=False,
            hold_back=6,
            missing="raise",
        )

        fitted_result = short_run_model.fit()
        fitted_results[subclass] = fitted_result

        validation_records.append(
            {
                "SubclassDescription": subclass,
                "Price_Lag": price_lag,
                "Exchange_Rate_Lag": exchange_rate_lag,
                "Exported_BIC": selection_row.BIC,
                "Reestimated_BIC": fitted_result.bic,
                "Absolute_BIC_Difference": abs(
                    selection_row.BIC - fitted_result.bic
                ),
                "BIC_Match": np.isclose(
                    selection_row.BIC,
                    fitted_result.bic,
                    rtol=1e-10,
                    atol=1e-8,
                ),
                "Observations": int(fitted_result.nobs),
            }
        )

    validation_table = (
        pd.DataFrame(validation_records)
        .sort_values("SubclassDescription")
        .reset_index(drop=True)
    )

    return fitted_results, validation_table

In [36]:
# re-estimate symmetric short-run models
(
    symmetric_short_run_results,
    symmetric_short_run_validation,
) = fit_selected_short_run_models(
    data=econometric_data,
    selection_table=symmetric_short_run_selection,
    exogenous_columns=["ExchangeRate_Log_Change_Pct"],
)

# Re-estimate asymmetric short-run models
(
    asymmetric_short_run_results,
    asymmetric_short_run_validation,
) = fit_selected_short_run_models(
    data=econometric_data,
    selection_table=asymmetric_short_run_selection,
    exogenous_columns=[
        "Depreciation_Shock_Pct",
        "Appreciation_Shock_Pct",
    ],
)

print(
    "Symmetric short-run models re-estimated:",
    len(symmetric_short_run_results),
)
print(
    "Asymmetric short-run models re-estimated:",
    len(asymmetric_short_run_results),
)

d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be use

Symmetric short-run models re-estimated: 46
Asymmetric short-run models re-estimated: 46


d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be use

In [37]:
# validate the reconstructed short-run models
symmetric_short_run_validation["Model"] = "Symmetric"
asymmetric_short_run_validation["Model"] = "Asymmetric"

short_run_model_validation = pd.concat(
    [
        symmetric_short_run_validation,
        asymmetric_short_run_validation,
    ],
    ignore_index=True,
)

short_run_reproduction_summary = (
    short_run_model_validation
    .groupby("Model", observed=True)
    .agg(
        Models=("SubclassDescription", "size"),
        BIC_Matches=("BIC_Match", "sum"),
        Maximum_BIC_Difference=(
            "Absolute_BIC_Difference",
            "max",
        ),
        Minimum_Observations=("Observations", "min"),
        Maximum_Observations=("Observations", "max"),
    )
    .reset_index()
)

display(short_run_reproduction_summary)

print(
    "All short-run BIC values reproduced:",
    short_run_model_validation["BIC_Match"].all(),
)

,Model,Models,BIC_Matches,Maximum_BIC_Difference,Minimum_Observations,Maximum_Observations
0,Asymmetric,46,46,0.000000,99,99
1,Symmetric,46,46,0.000000,99,99


All short-run BIC values reproduced: True


## Residual Serial-Correlation Diagnostics

Residual serial correlation occurs when model errors remain related across time. Its presence may indicate that the selected lag structure has not fully captured the dynamics in food-price inflation.

The Ljung–Box test is applied at 12 lags because the data are monthly. The test therefore assesses whether the residual autocorrelations through one annual cycle are jointly equal to zero.

The hypotheses are:

- **Null hypothesis:** The residuals are not serially correlated.
- **Alternative hypothesis:** The residuals exhibit serial correlation.

A p-value below 0.05 is treated as evidence against the null hypothesis.

The diagnostic is applied to the selected symmetric and asymmetric level models, as well as the corresponding short-run models. The level-model diagnostics use the ARDL representations because their UECM forms are reparameterisations of the same underlying models.

In [38]:
# Configure the Serial-Correlation Tests

serial_correlation_lags = 12

diagnostic_model_collections = {
    "Symmetric level": symmetric_level_ardl_results,
    "Asymmetric level": asymmetric_level_ardl_results,
    "Symmetric short run": symmetric_short_run_results,
    "Asymmetric short run": asymmetric_short_run_results,
}

model_collection_summary = pd.DataFrame(
    {
        "Model": diagnostic_model_collections.keys(),
        "Models": [
            len(model_results)
            for model_results in diagnostic_model_collections.values()
        ],
    }
)

display(model_collection_summary)

,Model,Models
0,Symmetric level,46
1,Asymmetric level,46
2,Symmetric short run,46
3,Asymmetric short run,46


In [39]:
# run the Ljung–Box Tests

def run_serial_correlation_tests(model_collections, test_lag=12):
    records = []

    for model_name, model_results in model_collections.items():
        for subclass, fitted_model in model_results.items():
            residuals = np.asarray(fitted_model.resid, dtype=float)
            residuals = residuals[np.isfinite(residuals)]

            ar_lags = getattr(fitted_model.model, "ar_lags", None)
            model_df = len(ar_lags) if ar_lags is not None else 0

            test_output = acorr_ljungbox(
                residuals,
                lags=[test_lag],
                model_df=model_df,
                return_df=True,
            )

            selected_result = test_output.iloc[0]
            test_statistic = float(selected_result["lb_stat"])
            p_value = float(selected_result["lb_pvalue"])

            records.append(
                {
                    "SubclassDescription": subclass,
                    "Model": model_name,
                    "Residual_Observations": len(residuals),
                    "AR_Lags": model_df,
                    "Test_Lag": test_lag,
                    "Degrees_of_Freedom": test_lag - model_df,
                    "Ljung_Box_Statistic": test_statistic,
                    "P_Value": p_value,
                    "No_Serial_Correlation": p_value >= 0.05,
                }
            )

    return pd.DataFrame(records)


serial_correlation_results = run_serial_correlation_tests(
    diagnostic_model_collections,
    test_lag=serial_correlation_lags,
)

print(
    "Serial-correlation tests completed:",
    len(serial_correlation_results),
)

print(
    "Missing diagnostic p-values:",
    serial_correlation_results["P_Value"].isna().sum(),
)

Serial-correlation tests completed: 184
Missing diagnostic p-values: 0


In [40]:
# summarise the Diagnostic Results

serial_correlation_summary = (
    serial_correlation_results
    .groupby("Model", as_index=False)
    .agg(
        Models=("SubclassDescription", "size"),
        Passed=("No_Serial_Correlation", "sum"),
    )
)

serial_correlation_summary["Failed"] = (
    serial_correlation_summary["Models"]
    - serial_correlation_summary["Passed"]
)

serial_correlation_summary["Pass_Rate_Pct"] = (
    100
    * serial_correlation_summary["Passed"]
    / serial_correlation_summary["Models"]
)

display(serial_correlation_summary)

,Model,Models,Passed,Failed,Pass_Rate_Pct
0,Asymmetric level,46,37,9,80.434783
1,Asymmetric short run,46,40,6,86.956522
2,Symmetric level,46,37,9,80.434783
3,Symmetric short run,46,41,5,89.130435


In [41]:
# identify Models with Serial Correlation

serial_correlation_failures = (
    serial_correlation_results.loc[
        ~serial_correlation_results["No_Serial_Correlation"]
    ]
    .sort_values(["Model", "P_Value"])
    .reset_index(drop=True)
)

print(
    "Models with evidence of serial correlation:",
    len(serial_correlation_failures),
)

if serial_correlation_failures.empty:
    print("No models failed the Ljung–Box test at the 5% level.")
else:
    display(serial_correlation_failures)

Models with evidence of serial correlation: 29


,SubclassDescription,Model,Residual_Observations,AR_Lags,Test_Lag,Degrees_of_Freedom,Ljung_Box_Statistic,P_Value,No_Serial_Correlation
0,"Macaroni, noodles, couscous and similar pasta ...",Asymmetric level,99,1,12,11,28.208724,0.003007,False
1,Fruit and vegetable juices,Asymmetric level,99,1,12,11,26.194036,0.006072,False
2,Baby food,Asymmetric level,99,2,12,10,24.332343,0.006765,False
3,Tubers,Asymmetric level,99,2,12,10,23.809259,0.008123,False
4,Milk,Asymmetric level,99,1,12,11,22.031833,0.024128,False
5,Other milk and cream,Asymmetric level,99,1,12,11,21.827834,0.025741,False
6,Coffee and coffee substitutes,Asymmetric level,99,1,12,11,21.659422,0.027149,False
7,Other non-alcoholic beverages,Asymmetric level,99,1,12,11,21.632521,0.027380,False
8,Other food products n.e.c.,Asymmetric level,99,2,12,10,18.517395,0.046838,False
9,"Macaroni, noodles, couscous and similar pasta ...",Asymmetric short run,99,3,12,9,28.162237,0.000896,False


### Serial-Correlation Results

Most selected models do not exhibit significant residual serial correlation at the 5% level. The pass rates range from 80.4% for the level specifications to 89.1% for the symmetric short-run specifications.

The diagnostic failures are concentrated in particular food subclasses rather than affecting all models uniformly. Several subclasses fail under more than one specification, indicating that some category-specific price dynamics may not be fully captured by the BIC-selected lag structures.

These failures do not invalidate the complete econometric analysis. However, coefficient inference from affected specifications should be treated cautiously because residual dependence can affect conventional standard errors and hypothesis tests.

The most important next check is whether the cointegrated level models used for the primary long-run results satisfy the serial-correlation diagnostic.

In [42]:
# check the Primary Long-Run Models

level_serial_correlation = serial_correlation_results.loc[
    serial_correlation_results["Model"].isin(
        ["Symmetric level", "Asymmetric level"]
    )
].copy()

level_serial_correlation["Model"] = (
    level_serial_correlation["Model"]
    .replace(
        {
            "Symmetric level": "ARDL",
            "Asymmetric level": "NARDL",
        }
    )
)

primary_long_run_models = final_bounds_results.loc[
    final_bounds_results["Final_Decision"].eq("Cointegration"),
    [
        "SubclassDescription",
        "Model",
        "Bounds_Statistic",
        "Final_Decision",
    ],
].copy()

primary_long_run_serial_diagnostics = primary_long_run_models.merge(
    level_serial_correlation[
        [
            "SubclassDescription",
            "Model",
            "Ljung_Box_Statistic",
            "P_Value",
            "No_Serial_Correlation",
        ]
    ],
    on=["SubclassDescription", "Model"],
    how="left",
    validate="one_to_one",
)

display(
    primary_long_run_serial_diagnostics.sort_values(
        ["Model", "P_Value"]
    ).reset_index(drop=True)
)

primary_long_run_serial_summary = (
    primary_long_run_serial_diagnostics
    .groupby("Model", as_index=False)
    .agg(
        Models=("SubclassDescription", "size"),
        Passed=("No_Serial_Correlation", "sum"),
    )
)

primary_long_run_serial_summary["Failed"] = (
    primary_long_run_serial_summary["Models"]
    - primary_long_run_serial_summary["Passed"]
)

display(primary_long_run_serial_summary)

,SubclassDescription,Model,Bounds_Statistic,Final_Decision,Ljung_Box_Statistic,P_Value,No_Serial_Correlation
0,"Chocolate, cocoa, and cocoa-based food products",ARDL,7.502411,Cointegration,16.192063,0.134149,True
1,Yoghurt and similar products,ARDL,7.295599,Cointegration,11.420135,0.325735,True
2,"Other vegetables, fresh or chilled",ARDL,7.372386,Cointegration,7.387174,0.688457,True
3,"Fruit-bearing vegetables, fresh or chilled",NARDL,8.346412,Cointegration,16.665141,0.054225,True
4,"Chocolate, cocoa, and cocoa-based food products",NARDL,6.235160,Cointegration,17.906933,0.083765,True
5,Cereals,NARDL,6.620570,Cointegration,17.180201,0.102657,True
6,"Dates, figs and tropical fruits, fresh",NARDL,9.173770,Cointegration,12.717140,0.239920,True
7,Yoghurt and similar products,NARDL,7.181472,Cointegration,12.577165,0.248283,True
8,"Other vegetables, fresh or chilled",NARDL,7.406674,Cointegration,6.700292,0.753404,True


,Model,Models,Passed,Failed
0,ARDL,3,3,0
1,NARDL,6,6,0


### Primary Long-Run Model Assessment

All nine bounds-supported level models pass the Ljung–Box test at the 5% significance level. This includes all three ARDL specifications and all six NARDL specifications retained by the sample-specific bounds tests.

The NARDL model for fruit-bearing vegetables has a p-value of 0.054, which is close to the 5% threshold. It is therefore retained but treated as a borderline diagnostic result.

Overall, the absence of significant residual serial correlation supports the dynamic specification of the models used for the primary long-run analysis. This finding does not override the separate error-correction assessment. In particular, the chocolate ARDL specification remains unsuitable for primary long-run interpretation because its adjustment coefficient was not statistically significant.